[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Creating Arrays


## What you will be able to do

Make the array you need before you have its values, with `np.zeros`, `np.ones`, `np.full`,
`np.arange`, `np.linspace` and `np.eye`, and say which type each one gives you.

## The idea

### The problem

The **Why NumPy** notebook started from readings that already existed as Python lists. Much of the
time you need an array before you have its values. A result to fill in station by station starts
as four empty places. The hours of the year are the numbers 0 to 8,759, and the start of each day is
every 24th of them. A temperature scale for a chart runs from -20 to 20 in steps of 5, and a
season's curve needs the fraction of the year that each hour has reached.

You could build each of those as a list and convert it: `np.array([0.0] * 4)`, or
`np.array(list(range(8760)))`. That works, and it builds every value as a Python object first, only
to copy it into an array straight after. It also stops at decimals, because `range` takes only
whole numbers, so a scale in steps of 0.25 needs a loop of its own.

### What an array constructor is

> An **array constructor** is a NumPy function that makes a new array from a description of it, a
> shape and a value to fill it with, or a start, a stop and a step, instead of from values you
> already have. Each constructor chooses a **dtype** for the array it makes, and takes a `dtype`
> argument to choose another.

### Why it works that way

`np.zeros`, `np.ones` and `np.full` take a **shape**, which is a single number for a row of values
or a tuple such as `(4, 8760)` for a grid. They fill every place with the same value, and they choose
`float64` unless told otherwise, because most numerical work is in decimals. A filled array is
something to write into, the way an empty list is something to append to.

`np.arange` is `range` for arrays. It counts from a start up to a stop, which it leaves out, in
steps of a size you give. Its dtype follows its arguments, so whole numbers give `int64` and a
decimal anywhere gives `float64`. `np.linspace` is for the other way of describing a scale. You give
the two ends and how many points you want, and it works out the step, with both ends included.

The surprise is how those two meet decimals. `arange` decides how many values to make by dividing
the distance by the step, in floating point, and a step such as 0.1 cannot be stored exactly. The
division can come out a hair over a whole number, and then the stop that `arange` is meant to leave
out appears at the end. `linspace` counts points, so it cannot make one too many.

### Where this shows up

Every computation that fills in results as it goes starts from `np.zeros` or `np.full`, which the
**Performance** notebook measures against growing an array one value at a time. `np.arange` gives
the positions that turn into dates and times, and `np.linspace` gives the evenly spaced inputs a
chart or a curve is drawn from, which the **Matplotlib, Deep Dive** guide uses throughout. The
**Linear Algebra** notebook starts from `np.eye`.

### What this notebook covers

- arrays from lists, and the dtype one decimal chooses
- `np.zeros`, `np.ones` and `np.full`, and why `nan` is a better placeholder than zero
- a shape as one argument, and arrays shaped like another with `zeros_like` and `full_like`
- `np.arange` for whole numbers, and with a decimal step
- `np.linspace` for a number of points between two ends
- the readings' seasonal swing, built from `np.arange` in one line
- `np.eye`, and the dtype every constructor chose

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

print(np.zeros(3))
print(np.arange(0, 24, 6))
print(np.linspace(0, 1, 5))
```

```
[0. 0. 0.]
[ 0  6 12 18]
[0.   0.25 0.5  0.75 1.  ]
```

Three zeros, written `0.` because they are decimals. The hours from 0 in steps of 6, stopping before
24, as whole numbers. Five points from 0 to 1 with both ends included, which puts them 0.25 apart.

## Setup

Four imports, and the year of readings as lists and as an array.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings
- `warnings` catches the warning that one of the Common errors gives, so that its message prints
  without the path of a file inside NumPy

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained. `readings` is the same values
as one array of shape (4, 8760), with `nan` for Svalbard's silent day.

In [1]:
import math
import warnings
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan

print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### Arrays from lists, and the dtype one decimal chooses

`np.array` looks at every value before it chooses a dtype, and one decimal is enough to make the
whole array decimals. A list of lists of the same length makes a grid:

In [2]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2.5, 3]).dtype)
print(np.array([1, 2.5, 3]))
print(np.array([[1, 2, 3], [4, 5, 6]]).shape)

int64
float64
[1.  2.5 3. ]
(2, 3)


### Zeros to fill in

`np.zeros(4)` makes four places for a result, one per station. Here a loop fills them with each
station's mean, which **The axis Argument** notebook does without the loop:

In [3]:
means = np.zeros(4)
print(means)

for i in range(4):
    means[i] = np.nanmean(readings[i])

print(means.round(2))

[0. 0. 0. 0.]
[ 8.    6.5  -4.48  3.5 ]


### Ones, and a value of your choosing

`np.ones` fills with 1, and `np.full` with any value you give it. `nan` is the better placeholder for
results not filled in yet, because 0.0 is a temperature a station can really report, and an
unfilled zero would pass for a reading:

In [4]:
print(np.ones(3))
print(np.full((2, 3), 7))
print(np.full(4, np.nan))

[1. 1. 1.]
[[7 7 7]
 [7 7 7]]
[nan nan nan nan]


`np.full((2, 3), 7)` holds whole numbers, because the value given was one. `np.full(4, np.nan)` holds
decimals, because `nan` is a decimal.

### A shape as one argument

A grid's shape goes in as one tuple, `(rows, columns)`. Here is an empty result shaped like the
readings, one place for every station and hour:

In [5]:
grid = np.zeros((4, HOURS))

print(grid.shape, grid.size, grid.dtype)

(4, 8760) 35040 float64


### Arrays shaped like another

`np.zeros_like` and `np.full_like` copy another array's shape and dtype, so the result always fits
the array it was made from:

In [6]:
blank = np.full_like(readings, np.nan)

print(blank.shape, blank.dtype)
print(np.isnan(blank).sum(), "of", blank.size, "are nan")

(4, 8760) float64
35040 of 35040 are nan


The dtype is copied too, which is what you want here and a trap with an array of whole numbers. The
fourth of the Common errors below shows what happens to `nan` then.

### Counting with arange

`np.arange(HOURS)` is every hour's position in the year, and a step of 24 picks the first hour of
every day:

In [7]:
hours = np.arange(HOURS)
day_starts = np.arange(0, HOURS, 24)

print(hours[:5], hours[-1], hours.dtype, hours.size)
print(day_starts[:3], day_starts[-1], day_starts.size)
print(np.arange(1, 13))

[0 1 2 3 4] 8759 int64 8760
[ 0 24 48] 8736 365
[ 1  2  3  4  5  6  7  8  9 10 11 12]


`hours` stops at 8,759, one before `HOURS`, as `range` would. `day_starts` has 365 values, the last
of them 8,736, the first hour of 31 December. `np.arange(1, 13)` is the twelve months.

### arange with a decimal step

A decimal step is allowed, and a decimal anywhere makes the result decimals:

In [8]:
print(np.arange(0, 1, 0.25))
print(np.arange(0, 2.0))

[0.   0.25 0.5  0.75]
[0. 1.]


This one stopped where it should. The third of the Common errors below shows one that does not,
which is why the next section exists.

### linspace for a number of points between two ends

`np.linspace(start, stop, number)` gives that many points, both ends included. A scale from -20 to 20
in steps of 5 is nine points:

In [9]:
print(np.linspace(-20, 20, 9))
print(np.linspace(0, 1, 5))
print(np.linspace(0, 24, 4, endpoint=False))

[-20. -15. -10.  -5.   0.   5.  10.  15.  20.]
[0.   0.25 0.5  0.75 1.  ]
[ 0.  6. 12. 18.]


`endpoint=False` leaves the stop out, as `arange` does, so four points from 0 that stop short of 24
land on 0, 6, 12 and 18.

### The readings' seasonal swing in one line

`year_of_readings` in Setup builds each reading in a loop, and part of each one is a seasonal swing,
`-math.cos(2 * math.pi * (n - 400) / HOURS)` for hour `n`. With `hours` from `np.arange`, the same
swing for all 8,760 hours is one line, `np.cos` being the array version of `math.cos`:

In [10]:
season = -np.cos(2 * np.pi * (hours - 400) / HOURS)

print(season.shape)
print(season[:3].round(3))
print(season.min().round(3), np.argmin(season), season.max().round(3), np.argmax(season))

(8760,)
[-0.959 -0.959 -0.96 ]
-1.0 400 1.0 4780


The swing is lowest at hour 400, in the middle of January, and highest half a year later, at hour
4,780. The **Vectorized Math** notebook builds the whole formula this way.

### The identity matrix, and the dtype every constructor chose

`np.eye(3)` is the identity matrix, ones on the diagonal and zeros everywhere else, which the
**Linear Algebra** notebook needs. Last, every constructor in this notebook and the dtype it chose:

In [11]:
print(np.eye(3, dtype=int))

made = {
    "np.array([1, 2, 3])": np.array([1, 2, 3]),
    "np.array([1, 2.5, 3])": np.array([1, 2.5, 3]),
    "np.zeros(4)": np.zeros(4),
    "np.full((2, 3), 7)": np.full((2, 3), 7),
    "np.arange(HOURS)": np.arange(HOURS),
    "np.arange(0, 1, 0.25)": np.arange(0, 1, 0.25),
    "np.linspace(0, 1, 5)": np.linspace(0, 1, 5),
    "np.eye(3)": np.eye(3),
}
for text, array in made.items():
    print(f"{text:24} {array.dtype}")

[[1 0 0]
 [0 1 0]
 [0 0 1]]
np.array([1, 2, 3])      int64
np.array([1, 2.5, 3])    float64
np.zeros(4)              float64
np.full((2, 3), 7)       int64
np.arange(HOURS)         int64
np.arange(0, 1, 0.25)    float64
np.linspace(0, 1, 5)     float64
np.eye(3)                float64


## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/02-creating-arrays-solutions.ipynb).

**1.** Make an array of 365 zeros, one for each day of the year, and print its shape and dtype.

In [12]:
# your code here

**2.** Make the hours of one day, 0 to 23, with `np.arange`, and print the last five.

In [13]:
# your code here

**3.** Make 11 evenly spaced temperatures from -10 to 10, both ends included, and print them.

In [14]:
# your code here

**4.** Make an array shaped like `readings` and filled with `nan`, and print its shape and how many of its values are `nan`.

In [15]:
# your code here

**5.** Make a 3 by 3 identity matrix of whole numbers, and print it.

In [16]:
# your code here

## Common errors

### TypeError: Cannot interpret '8760' as a data type

In [17]:
np.zeros(4, HOURS)

TypeError: Cannot interpret '8760' as a data type

`np.zeros` takes the shape as its first argument and the dtype as its second, so `HOURS`, 8,760, was
read as a dtype. A grid's shape is one tuple: `np.zeros((4, HOURS))`.

### TypeError: array() takes from 1 to 2 positional arguments but 3 were given

In [18]:
np.array(8.2, 7.9, 7.5)

TypeError: array() takes from 1 to 2 positional arguments but 3 were given

`np.array` takes the values as one argument, a list or a tuple, so three separate numbers are two
too many. `np.array([8.2, 7.9, 7.5])` passes them as one list.

### No error, and an arange that includes its stop

In [19]:
steps = np.arange(1, 1.3, 0.1)

print(steps)
print(steps.size)
print((1.3 - 1) / 0.1)                                # how many steps arange worked out
print(np.linspace(1, 1.2, 3))

[1.  1.1 1.2 1.3]
4
3.0000000000000004
[1.  1.1 1.2]


`np.arange(1, 1.3, 0.1)` was meant to stop before 1.3, and returned it. Neither 0.1 nor 1.3 can be
stored exactly, so the number of steps came out as 3.0000000000000004, a hair over 3, and `arange`
rounded it up to 4 values. For decimals, give `linspace` the number of points instead, as in
`np.linspace(1, 1.2, 3)`, which stops at 1.2.

### RuntimeWarning: invalid value encountered in cast

In [20]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    unfilled = np.full_like(hours, np.nan)

print(f"{caught[0].category.__name__}: {caught[0].message}")
print(unfilled.dtype, unfilled[:3])

int64 [-9223372036854775808 -9223372036854775808 -9223372036854775808]


`hours` holds whole numbers, so `np.full_like` made an `int64` array, and an `int64` cannot hold
`nan`. NumPy cast `nan` to a whole number anyway and warned, and what `nan` becomes in that cast is
not defined. In this run it became -9223372036854775808, the most negative `int64`. Pass the dtype:
`np.full_like(hours, np.nan, dtype=float)`. The cell catches the warning with `warnings` so that its
message prints on its own, without the file inside NumPy that raised it.

### No error, and an empty array from arange

In [21]:
countdown = np.arange(24, 0)

print(countdown, countdown.size)

[] 0


`arange` counts up unless the step says otherwise, and from 24 there is nothing to count up to 0,
so the result is empty and nothing warns you. A countdown needs a negative step:
`np.arange(24, 0, -1)`.

## Recap

- Make an array to fill in with `np.zeros`, `np.ones` or `np.full`, giving a grid's shape as one
  tuple
- Use `nan` as the placeholder for values not known yet, and copy another array's shape with
  `zeros_like` and `full_like`
- Count whole numbers with `np.arange`, and give `np.linspace` a number of points for decimals
- Say which dtype each constructor chooses, and pass `dtype=` to choose another
- Build a whole column of results, such as the seasonal swing, from `np.arange` in one line

## What is next

The **dtype** notebook looks at the types an array can hold, and at the number that overflows its
type without any warning.

---

&#8592; **Previous:** [Why NumPy](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/01-why-numpy.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
